# Laboratorio 1 : Introducción Blockchain

In [1]:
from dataclasses import dataclass, field
from typing import *
import rsa
import hashlib
import time

# Transacción

A continuación implementaremos una **transacción.** Una transacción requiere:
- Dirrección del emisor
- Dirección del receptor
- Cantidad de tokens que se están transfiriendo.

Una transacción es el elemento básico que se registra en la blockchain. Por ejemplo, si Chelo le quiere enviar 10 tokens a Sergio, eso se registra como una transacción.

**¿Por qué necesitamos firmar una transacción (método sign)?**

Porque en una blockchain descentralizada y sin confianza entre usuarios, no puedes simplemente decir Chelo envió 10 tokens a Sergio y creerlo. Necesitamos verificar que Chelo realmente autorizó esa operación.

Eso se logra con:

    Una clave privada: Que solo tiene Chelo y usa para firmar.

    Una clave pública: Que todos conocen, y se usa para validar esa firma.

Así, la transacción firmada prueba que fue autorizada por el propietario legítimo


**¿ Cómo se verifica ?**

rsa.verify


- message: Es el contenido original que se firmó, en este caso:

    self.get_header() → que devuelve un bytes que representa el resumen de la transacción (emisor + receptor + valor).

- signature: La firma digital de ese mensaje, generada previamente usando rsa.sign() con la clave privada del emisor.

* pub_key: La clave pública del emisor. Sirve para comprobar que la firma realmente fue generada por su clave privada.

In [2]:
@dataclass
class Transaction:
    #Clave publica del emisor y receptor
    sender_address: rsa.PublicKey
    recipient_address: rsa.PublicKey
    value: float
    #Firma digital (vacía al inicio).
    signature: bytes = field(default=b'')


    #Firma la transacción con la clave privada del emisor.
    def sign(self, sender_private_key: rsa.PrivateKey) -> None:
        """Toma una clave privada que el remitente nunca debe compartir,
        y firma esta transacción para verificar que desea transferir tokens.
        Se firman los atributos centrales de la transacción usando la
        clave privada. Usaremos SHA-256 como el algoritmo para firmar."""
        self.signature = rsa.sign(self.get_header(), sender_private_key, 'SHA-256')
        #Llama a validate() para asegurarse de que la firma es válida.
        self.validate()

    #Prepara los datos esenciales de la transacción (emisor, receptor, valor) para ser firmados.
    #Crea un resumen codificado de los datos a firmar.

    def get_header(self) -> bytes:
        """Proporciona la información básica necesaria para firmar la transacción"""
        return (str(self.sender_address) + str(self.recipient_address) + str(self.value)).encode()

    #Verifica que la firma sea válida usando la clave pública del emisor.
    #Usa la clave pública del emisor para verificar que la firma es auténtica.
    #Asegura que nadie haya modificado los datos después de la firma.

    #rsa.verify(
    #message: bytes,
    #signature: bytes,
    #pub_key: rsa.PublicKey
    #)


    def validate(self) -> None:
        """Valida la integridad de la transacción usado RSA para verificar que se ha
        firmado correctamente. Obtiene la información másica sobre la transacción,
        la firma usando la clave privada del remitente, y la clave pública para validar
        la firma. La clase pública es la dirección del emisor."""
        rsa.verify(self.get_header(), self.signature, self.sender_address)


    #Si algo falla (por ejemplo, la firma no coincide con el mensaje o con la clave pública), lanza una excepción:  rsa.VerificationError

# Bloques

Ahora que hemos definido una transacción, no podemos centrar en los **bloques.** Éstos, necesitan:
- Identificador del bloque
- Marca temporal
- Lista de las transacciones incluidas en el bloque
- Hash del bloque anterior de la cadena, importante para asegurar la inmutabilidad.
- Hash del bloque actual, de nuevo, para asegurar inmutabilidad.
- Nonce, usado para ayudar en la implementación del algoritmo distribuido de consenso de la confianza.



In [3]:
@dataclass
class Block:
    #Identificador del bloque (por ejemplo, bloque 0 es el génesis).
    num: int
    # Fecha y hora de creación (formato float de time.time()).
    timestamp: float
    # Hash del bloque anterior: permite encadenarlos.
    prev_block_hash: str
    # Lista de objetos Transaction que contiene el bloque.
    transactions: List[Transaction]
    # Hash del bloque actual, se calcula en hash().
    block_hash: str = field(default="")
    # El nonce es un número que se ajusta en cada intento para encontrar un hash válido que cumpla con las reglas del algoritmo de consenso
    nonce: int = field(default=0)


    #Este hash es como una huella digital única del bloque.
    def hash(self) -> str:
        """Usa SHA-256 para codificar el encabezado del bloque, utilizando
        los atributos principales. Guarda el hash en el bloque para su uso
        posterior."""
        self.block_hash = hashlib.sha256(self.get_header()).hexdigest()
        return self.block_hash

    #Junta los datos esenciales del bloque: número, timestamp, hash anterior, lista de transacciones y nonce.
    #Los convierte en texto plano (str(...)) y luego a bytes (.encode()), que es lo que necesita SHA-256.
    def get_header(self) -> bytes:
        """Devuelve una cadena que representa los atributos principales que
        identifican de forma única el bloque y permite vincularlo al bloque
        anterior, formando la cadena. Estos atributos incluyen: número de bloque
        marca de tiempo, hash del bloque anterior, transancciones, y nonce."""
        return (str(self.num) + str(self.timestamp) + str(self.prev_block_hash) + str(self.transactions) +
                str(self.nonce)).encode()
    #Llama a self.hash() para recalcular el hash del bloque.
    #Pasa ese hash a una función de prueba de trabajo (proof_of_work_func) que comprueba si el bloque es válido.
    #Delegas la validación del bloque al algoritmo de consenso que uses (como Proof of Work, PoS)


    def validate(self, proof_of_work_func: Callable[[str], bool]) -> bool:
        """Dada una función, nos aseguramos que el bloque tenga el hash correcto,
        de acuerdo al algoritmo de consenso acordado."""
        return proof_of_work_func(self.hash())

Un algoritmo de consenso es el conjunto de reglas que permite que todos los nodos de una blockchain acuerden qué bloques son válidos, sin necesidad de confiar entre ellos.

Porque en una red distribuida (sin autoridad central), necesitamos una forma segura de:

- Validar qué bloques se aceptan.

- Evitar fraudes o doble gasto.

- Asegurar que todos los nodos tengan la misma versión de la verdad.

# Nodo Blockchain

Por último, una vez tenemos las transacciones y los bloques implementados, debemos crear un objeto que permita salvaguardar la estructura de la cadena. A este objeto le llamaremos nodo de la cadena de bloques, y permitirá aceptar transacciones, minar bloques y validar.

Un nodo de la cadena de bloques debe:
- Disponer de una dirección para recibir los tokens de recompensa por minar nuevos bloques.
- Contener lista de los bloques aceptados en la cadena.
- Recibir y validar nuevas transacciones.
- Minar nuevos bloques, y por tanto, añadir nuevas transacciones a la cadena.
- Validar la cadena

In [4]:
REWARD_AMOUNT = 2.0

#Es la recompensa por minar un bloque. Cuando un nodo encuentra un bloque válido, recibe 2 tokens.
class BlockchainNode:


    #Inicializamos
    def __init__(self, miner_address: rsa.PublicKey) -> None:
        #    La dirección del minero (clave pública).
        self.miner_address: rsa.PublicKey = miner_address
        # Lista de bloques (self.blocks)
        self.blocks: List[Block] = []
        #Lista de transacciones pendientes (self.pending_transactions)
        self.pending_transactions: List[Transaction] = []
        #Función de consenso → una lambda que define el Proof of Work (hash debe terminar en '00000').
        self.proof_of_work_func: Callable[[str], bool] = lambda x: x.endswith('00000')
        #Mina el primer bloque automáticamente (self.mine_block()), que actúa como bloque génesis.
        self.mine_block()

    #La función submit_transaction(transaction) sirve para recibir, validar y almacenar una transacción pendiente en un nodo de la blockchain, antes de que sea incluida en un bloque.

    def submit_transaction(self, transaction: Transaction) -> None:
        """Nos permite enviar una nueva transacción a la cadena. Los usuarios
        finales envian transacciones y no se preocupan sobre los bloques, perse.
        Esta función toma una transacción firmada y la validará. Posteriomente,
        comprueba que hay suficiente balance para realizar la transferencia. Si
        todo se cumple, se añadirá la transacción a la lista a considerar cuando se
        mine el siguiente bloque."""

        # Asegúrese de que la transacción esté debidamente firmada por la clave privada
        transaction.validate()

        # Asegúrese de que existan los fondos para la transacción solicitada
        balance = self.get_balance(transaction.sender_address)

        if balance < transaction.value:
            raise Exception("Insufficient funds!")

        # Transaction checks out--add to the list of our pending
        # transactions!

        self.pending_transactions.append(transaction)

    def mine_block(self) -> None:
        """Esta función agrupa todas las transacciones pendientes y
        mina un nuevo bloque. Mediante minar podemos añadir un nuevo
        bloque a la cadena. Aquí entra en juego el algoritmo de consenso
        descentralizado. Usamos un algoritmo de prueba de trabajo que es
        computacionalmente caro. Esto permite evitar llenar la cadena P2P
        de transacciones y bloques inválidos. Al ser tan costoso, dificulta
        reescribir la historia, haciendo que la cadena de bloques sea más
        segura de este tipo de ataques. Esta función minará un nuevo bloque
        y lo añadirá automáticamente a la cadena. Los nodos que extraen con
        éxito un nuevo bloque obtiene un premio. De esta forma comienzan a
        circular nuevos tokens en la cadena e incentiva a que la gente se una
        a la red."""

        # Si la blockchain aún no tiene bloques, este será el bloque génesis (el primero).
        # En ese caso, no hay hash previo, así que se pone un valor simbólico.
        if len(self.blocks) <= 0:
            prev_hash = "-"
            next_num = 0
        else:

        # Si ya hay bloques, obtenemos el último bloque de la cadena
         # para usar su hash y su número como referencia del nuevo bloque.

            prev_block = self.blocks[-1]
            prev_hash = prev_block.block_hash
            next_num = prev_block.num + 1

        # Obtenga el bloque anterior en la cadena y use su número
        # y hash para incorporar al nuevo bloque.

        # Creamos un nuevo bloque con:
        # - el número correspondiente
        # - la marca de tiempo actual
        # - el hash del bloque anterior
        # - una copia de las transacciones pendientes


        new_block = Block(next_num, time.time(), prev_hash, self.pending_transactions.copy())

        # Agregue una transacción de recompensa al final del bloque para este nodo

        reward = Transaction(None, self.miner_address, REWARD_AMOUNT)

        new_block.transactions.append(reward)

        # Ejecutamos el algoritmo Proof of Work para encontrar un nonce válido
        self.execute_pow(new_block)

        # Agrega el bloque a la cadena
        self.blocks.append(new_block)

        # Vaciamos la lista de transacciones pendientes, ya que ya se han incluido en el nuevo bloque
        self.pending_transactions.clear()

        # Confirmamos visualmente que el bloque ha sido minado con éxito
        print(f"Successfully mined new block {new_block.num}")


    #Es el método que simula el algoritmo de consenso Proof of Work (PoW) dentro de tu nodo de blockchain.
    def execute_pow(self, block: Block) -> None:
        """Utilizando la prueba de trabajo lambda definida,
        incrementamos el nonce en el bloque hasta satisfagamos
        la afirmación de la lambda. Tanto la iteracción como el
        hashing conllevan mucho tiempo y mucho consumo de CPU.
        Finalmente, este método devolverá un nonce y hash que cumple
        los requisitos del algoritmo de consenso y se puede comprobar
        facilmente."""

        # Imprime éxito
        block.nonce = 0
        #Verifica si ese hash no cumple la condición
        while not block.validate(self.proof_of_work_func):
            block.nonce += 1
        #Ejemplos
        #Intento 1 → hash = "12aef000a"
        #Intento 2 → hash = "22aefc001"
        #Intento 98432 → hash = "00000abcde"

        print(f"Successfully mined new block {block.num} with nonce {block.nonce}")

  #Este método verifica la integridad completa de la cadena de bloques del nodo. Es como una auditoría criptográfica para asegurarse de que:
  # Nadie ha modificado bloques anteriores.
  # Todos los bloques respetan el algoritmo de consenso (Proof of Work).


    def validate_chain(self) -> None:
        """Verifica que la cadena es criptográficamente sólida y no ha
        sido modificada. También asegura que todos los bloques cumplen
        con el algoritmo de consenso. Esta función es mucho más rápida
        que minar un nuevo bloque. Minar un bloque es difícil pero la
        verificación es muy sencilla, algo esencial para asegurar la
        integridad de la red."""

        # Se empieza con el hash ficticio del bloque génesis (no tiene anterior)


        prev_hash = "-"

        #Recorremos todos los bloques de la cadena
        for b in self.blocks:
            # Verificamos que el bloque actual apunte correctamente al anterior
            if b.prev_block_hash != prev_hash:
                raise Exception("Previous hashes don't match")
          # Verificamos que el hash del bloque cumple con el algoritmo de consenso (ej. termina en '00000')
            elif not b.validate(self.proof_of_work_func):
                raise Exception("Something is wrong with PoW hash")
            # Si todo está correcto, actualizamos el hash para el siguiente bloque
            prev_hash = b.block_hash

       # Si la función termina sin lanzar excepciones, la cadena es válida
        print(f"Successfully validated chain of size {len(self.blocks)}")

   #Este método calcula el saldo total de una dirección pública (clave pública) recorriendo toda la cadena de bloques y observando todas las transacciones donde aparece esa dirección.
   #No existe una “cuenta central”, sino que el saldo se reconstruye históricamente revisando todas las transacciones

    def get_balance(self, address: rsa.PublicKey) -> float:
        """Este método proporciona una manera fácil de recorrer la cadena
        para averiguar el saldo de la dirección dada. Bitcoin no trabaja de
        esta manera, pero otras sí. Cumple con nuestros requerimientos.
        Devuelve el saldo dada un dirección, devolviendo 0.0 si la dirección
        no ha sido vista en la cadena antes."""

        # Comience el saldo en 0 y recorra todos los bloques
        # y sus transacciones, buscando lo dado
        # dirección y realizar un seguimiento del saldo en consecuencia.
        balance = 0.0

        # Recorre cada bloque de la cadena
        for b in self.blocks:
            # Recorre cada transacción dentro del bloque
            for t in b.transactions:
                # Si la dirección es el remitente, resta el valor de la transacción
                if t.sender_address == address:
                    balance -= t.value
                # Si la dirección es el receptor, suma el valor recibido
                elif t.recipient_address == address:
                    balance += t.value

        return balance


Ahora podemos intentar crear transacciones y extraer algunos bloques. Primero necesitamos algunas claves públicas RSA para actuar como direcciones y permitir la firma. Podemos usar el módulo rsa incluido para crearlos para nosotros. Debido a los requisitos de firma RSA con SHA-256, necesitaremos claves de al menos 512 bits:

In [5]:
public_key, private_key = rsa.newkeys(512)
receiving_public_key, receiving_private_key = rsa.newkeys(512)

Ahora podemos usar nuestras claves pública y privada para realizar nuestra primera transacción:

In [6]:
t1 = Transaction(public_key, receiving_public_key, 0.5)
t1.sign(private_key)

Let's make another one. It's easy!

In [7]:
t2 = Transaction(public_key, receiving_public_key, 1.0)
t2.sign(private_key)

¿Qué sucede si intentamos crear una transacción para tomar tokens de una dirección para la que no tenemos la clave privada?

In [8]:
bad_t = Transaction(public_key, receiving_public_key, 1000.0)
bad_t.sign(receiving_private_key)

VerificationError: Verification failed

Ahora estamos listos para enviar nuestras transacciones a un nodo de cadena de bloques. Vamos a crear uno:

In [9]:
node = BlockchainNode(public_key)
node.submit_transaction(t1)

Successfully mined new block 0 with nonce 377443
Successfully mined new block 0


La transacción no se incluirá en la cadena de bloques hasta que se extraiga un bloque:

In [10]:
node.mine_block()

Successfully mined new block 1 with nonce 200948
Successfully mined new block 1


Podemos probar nuestra implementación y validar la integridad de la cadena usando el método validate_chain():

In [11]:
node.validate_chain()

Successfully validated chain of size 2


Intentemos piratear la cadena cambiando el valor de una transacción... ¡la inmutabilidad y la validez de las transacciones deberían violarse y la validación debería fallar!

In [12]:
hack = node.blocks[1].transactions[0]
orig_value = hack.value
hack.value = 100000.0
node.validate_chain()

Exception: Something is wrong with PoW hash

Restaurar la transacción a su estado original permitirá que la cadena vuelva a ser válida:

In [13]:
hack.value = orig_value
node.validate_chain()

Successfully validated chain of size 2


Ahora podemos mostrar cómo se aplican los "controles de gastos". Intentemos enviar una transacción que tenga una cantidad mayor que nuestro saldo:

In [14]:
#node.mine_block()
node.get_balance(public_key)
node.submit_transaction(t2)
node.mine_block()
node.get_balance(receiving_public_key)

Successfully mined new block 2 with nonce 1465744
Successfully mined new block 2


1.5